# Case Study A -- Quantitative Held-Out Inference on KT$_1$

Quantitative companion to the WAN topology-reconfiguration case study (paper Sec. V-A, Fig. 13).
A single event -- `Link_Down(Berlin, Hamburg)` = **L1** -- is studied, matching the figure. There is
no sweep over scenarios: this is a case study, and the claim is about *what the model infers from
partial information*, not about average behaviour over a benchmark.

## Ground truth

`demand_factor.ipynb` builds a two-level knowledge topology from the real SNDlib germany17 matrix:
KT$_0$ before the event, KT$_1$ after two-tier demand suppression. Its measured ratios are the
targets here, and both are graded in $[0,1]$ -- the same range as an LTN truth value:

* node: $\mathrm{DF}_1(v) = \mathrm{act}_1(v)/\mathrm{act}_0(v)$ -- retained node activity
* link: $\rho(l) = w_1(l)/w_0(l)$ -- retained link load (undirected, $\rho = 0$ on L1 by construction)

Grading matters. A binary "affected / unaffected" label on 26 links yields ~2 positives per
held-out set, so accuracy is dominated by the class prior and every threshold metric becomes a
statement about sigmoid calibration. The graded targets avoid that failure mode rather than
managing it.

## Protocol

`Node` is supervised on **all 17** DF$_1$ values; `Link` is supervised on a **revealed subset** of
$\rho$ and scored on the remainder. The asymmetry is the experiment: node-level demand knowledge
must reach link-level inference through the injected axioms, which is precisely the
knowledge-embedding claim. Two reveal policies are compared at 30 / 50 / 70 %:

* **Mode A (neighbourhood-first, ring-stratified)** -- reveal outward from L1 by hop ring, while
  reserving a fixed share of *every* ring as held-out. Without the reservation a 50 % reveal
  consumes hops 0-2 entirely and leaves a held-out set where 11 of 13 links have $\rho \ge 0.96$;
  a rank metric on a near-constant target is not informative.
* **Mode B (random)** -- uniform random reveal at the same fraction.

## Baselines

Reporting the model alone would not be interpretable, because **hop distance from L1 -- the injected
topological rule and nothing else -- already attains Spearman $|\rho| \approx 0.81$ against the link
target.** Four references are therefore evaluated on the identical held-out split:

| baseline | what it isolates |
| --- | --- |
| random scores | the evaluation protocol itself |
| hop distance | the topological rule alone |
| baseline load $w_0$ | demand alone, no logic |
| feature-only MLP | identical features and supervision, **axioms and `Node` removed** |

The last is the decisive one: if the full model does not beat it, the logic contributes nothing.

**The claim under test.** Hop distance orders the *rings*; it cannot order links *within* a ring.
At $\beta = 0.5$ the hop-3 ring spans $\rho \in [0.582, 1.000]$ -- L25 (Nuernberg-Stuttgart) sits at
0.582 while its ring-mates exceed 0.96, because of where traffic actually reroutes. Recovering that
within-ring ordering requires demand knowledge, and is what this experiment measures.

In [ ]:
import os, math, itertools, random
from collections import defaultdict
import numpy as np
import pandas as pd
import networkx as nx
import torch
import ltn
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import roc_auc_score

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("device:", device)

In [ ]:
# ----------------------------- Topology (germany17 / SNDlib, ref [27]) -----------------------------
raw_nodes = {
    "Hannover": (9.80, 52.39), "Frankfurt": (8.66, 50.14), "Hamburg": (10.08, 53.55),
    "Norden": (7.21, 53.60), "Bremen": (8.80, 53.08), "Berlin": (13.48, 52.52),
    "Muenchen": (11.55, 48.15), "Ulm": (9.99, 48.40), "Nuernberg": (11.08, 49.45),
    "Stuttgart": (9.12, 48.73), "Karlsruhe": (8.41, 49.01), "Mannheim": (8.49, 49.49),
    "Essen": (7.00, 51.44), "Dortmund": (7.48, 51.51), "Duesseldorf": (6.78, 51.22),
    "Koeln": (7.01, 50.92), "Leipzig": (12.38, 51.34),
}
raw_links = {
    "L1": ("Berlin","Hamburg"), "L2": ("Berlin","Hannover"), "L3": ("Berlin","Leipzig"),
    "L4": ("Bremen","Hamburg"), "L5": ("Bremen","Hannover"), "L6": ("Bremen","Norden"),
    "L7": ("Dortmund","Essen"), "L8": ("Dortmund","Hannover"), "L9": ("Dortmund","Koeln"),
    "L10": ("Dortmund","Norden"), "L11": ("Duesseldorf","Essen"), "L12": ("Duesseldorf","Koeln"),
    "L13": ("Frankfurt","Hannover"), "L14": ("Frankfurt","Koeln"), "L15": ("Frankfurt","Leipzig"),
    "L16": ("Frankfurt","Mannheim"), "L17": ("Frankfurt","Nuernberg"), "L18": ("Hamburg","Hannover"),
    "L19": ("Hannover","Leipzig"), "L20": ("Karlsruhe","Mannheim"), "L21": ("Karlsruhe","Stuttgart"),
    "L22": ("Leipzig","Nuernberg"), "L23": ("Muenchen","Nuernberg"), "L24": ("Muenchen","Ulm"),
    "L25": ("Nuernberg","Stuttgart"), "L26": ("Stuttgart","Ulm"),
}
def key(e):
    return tuple(sorted(e))

nodes     = list(raw_nodes.keys())
links     = [key(v) for v in raw_links.values()]
link_name = {key(v): k for k, v in raw_links.items()}
print(f"{len(nodes)} nodes, {len(links)} links")

In [ ]:
# ----------------------------- Configuration -----------------------------
CONFIG = dict(
    # --- event & ground truth ---
    down_link   = ("Berlin", "Hamburg"),   # = L1, the scenario shown in Fig. 13
    beta        = 0.5,                     # tier-2 suppression factor (see demand_factor.ipynb)

    # --- protocol ---
    reveal_fracs = [0.3, 0.5, 0.7],        # share of the 26 links revealed to the Link predicate
    modes        = ["hop", "random"],      # Mode A / Mode B
    min_hold_per_ring = 0.25,              # Mode A: share of every hop ring reserved as held-out
    seeds        = list(range(10)),

    # --- training ---
    epochs       = 1000,
    lr           = 0.001,
    lambda_sup   = 1.0,                    # weight of the graded supervision term vs the logic term
    hidden       = (32, 32),

    # --- evaluation ---
    tau_grid     = [0.99, 0.90, 0.75],     # "impacted" := rho < tau (a semantic choice, not a cutoff
                                           #  on the model's uncalibrated output)
    topk         = 3,

    # --- demand data (real SNDlib matrix required; see loader) ---
    demand_matrix = "demandMatrix-germany17-DFN-5min-20050215-0000.txt",
    demand_file   = None,                  # set explicitly to override the directory search

    quick_test   = False,                  # True -> 2 seeds x 200 epochs, one reveal fraction
)
if CONFIG["quick_test"]:
    CONFIG.update(epochs=200, seeds=[0, 1], reveal_fracs=[0.5])
CONFIG

In [ ]:
# ----------------------------- Demand loader (real SNDlib matrix required) -----------------------------
# A missing file is a hard error rather than a silent fallback. Substituting uniform all-pairs
# demand would leave the pipeline running and the numbers plausible while changing what the ground
# truth measures: with uniform demand the reroute structure collapses onto graph structure, and the
# targets would no longer carry information beyond the topological rule.
ARCHIVE_DIR = "directed-nobel-germany-DFN-aggregated-5min-over-1day-native"
DEMAND_DIR_CANDIDATES = [
    os.path.join(".", ARCHIVE_DIR),                 # extracted beside this notebook
    os.path.join(".", "Germany17", ARCHIVE_DIR),    # layout used by demand_factor.ipynb
    ".",
]

def parse_sndlib_demands(filepath):
    """SNDlib native format:  <id> ( <src> <dst> ) <routing_unit> <value> <max_path_length>"""
    out, in_section = [], False
    with open(filepath) as f:
        for line in f:
            s = line.strip()
            if s == "DEMANDS (":
                in_section = True;  continue
            if in_section and s == ")":
                in_section = False; continue
            if in_section and s:
                p = s.split()
                if len(p) >= 7 and p[1] == "(" and p[4] == ")":
                    out.append((p[2], p[3], float(p[6])))
    return out

def resolve_demand_file(cfg):
    if cfg.get("demand_file"):
        return cfg["demand_file"]
    for d in DEMAND_DIR_CANDIDATES:
        cand = os.path.join(d, cfg["demand_matrix"])
        if os.path.exists(cand):
            return cand
    return None

def load_demands(cfg):
    fp = resolve_demand_file(cfg)
    if not fp or not os.path.exists(fp):
        raise FileNotFoundError(
            f"Real SNDlib demand matrix {cfg['demand_matrix']!r} not found in "
            f"{DEMAND_DIR_CANDIDATES}.\n"
            "Download 'directed-nobel-germany-DFN-aggregated-5min-over-1day-native.tgz' from\n"
            "  https://sndlib.put.poznan.pl/dynamicmatrices.overview.action\n"
            "and extract it beside this notebook. This is the same file demand_factor.ipynb loads.")
    raw = parse_sndlib_demands(fp)
    unknown = {n for (s, t, _) in raw for n in (s, t)} - set(raw_nodes)
    if unknown:
        raise ValueError(f"demand file references nodes absent from the topology: {sorted(unknown)}")
    demands = [(s, t, v) for (s, t, v) in raw if v > 0 and s != t]
    if not demands:
        raise ValueError(f"no positive demands parsed from {fp} -- check the file format")
    print(f"Loaded {len(demands)} directed demands (total volume "
          f"{sum(v for *_, v in demands):.2f}) from {os.path.basename(fp)}")
    return demands, os.path.basename(fp)

DEMANDS, DEMAND_SOURCE = load_demands(CONFIG)

In [ ]:
# ----------------------------- KT0 / KT1 (identical pipeline to demand_factor.ipynb) -----------------------------
def geo_dist(a, b):
    (x1, y1), (x2, y2) = raw_nodes[a], raw_nodes[b]
    return math.hypot(x2 - x1, y2 - y1)

# Routing metric is the geographic edge weight w_g, exactly as in demand_factor.ipynb, so both
# notebooks route the same traffic over the same paths.
G0 = nx.Graph()
G0.add_nodes_from(nodes)
for (a, b) in links:
    G0.add_edge(a, b, w_g=geo_dist(a, b))
ROUTES = {(s, t): nx.shortest_path(G0, s, t, weight="w_g") for (s, t, v) in DEMANDS}

def compute_kt(demand_list):
    """Directed link loads w and node activity act (origin + destination + transit)."""
    w   = {d: 0.0 for pair in links for d in (pair, pair[::-1])}
    act = {n: 0.0 for n in raw_nodes}
    for (s, t, v) in demand_list:
        p = ROUTES[(s, t)]
        for (a, b) in zip(p, p[1:]):
            w[(a, b)] += v
        act[s] += v
        act[t] += v
        for n in p[1:-1]:
            act[n] += v
    return w, act

w0, act0 = compute_kt(DEMANDS)

DOWN     = key(CONFIG["down_link"])
down_set = frozenset(DOWN)
# R2/R3 (1-hop): links sharing an endpoint with the down link
affected_link_set = {frozenset(l) for l in links
                     if l[0] in DOWN or l[1] in DOWN} - {down_set}

def demand_factor_of(path, beta):
    """Two-tier suppression factor of one demand, decided by its (unchanged) route."""
    edges = {frozenset(e) for e in zip(path, path[1:])}
    if down_set in edges:            # tier 1: route crosses the down link
        return 0.0
    if edges & affected_link_set:    # tier 2: route crosses a 1-hop affected link
        return 1.0 - beta
    return 1.0

F  = {(s, t): demand_factor_of(ROUTES[(s, t)], CONFIG["beta"]) for (s, t, v) in DEMANDS}
w1, act1 = compute_kt([(s, t, v * F[(s, t)]) for (s, t, v) in DEMANDS])

# ---- evaluation targets, both graded in [0,1] ----
DF1 = {n: (act1[n] / act0[n] if act0[n] > 0 else 1.0) for n in nodes}
RHO = {}                                   # undirected: the Link predicate is symmetric
for (a, b) in links:
    num, den = w1[(a, b)] + w1[(b, a)], w0[(a, b)] + w0[(b, a)]
    RHO[key((a, b))] = num / den if den > 0 else 1.0

assert abs(RHO[DOWN]) < 1e-12, "rho on the down link must be exactly 0 by construction"
_r = np.array([RHO[l] for l in links])
print(f"rho over {len(links)} links : min {_r.min():.4f} max {_r.max():.4f} | "
      f"<1: {(_r < 1-1e-9).sum()} | distinct {len(set(np.round(_r,6)))}")
_d = np.array([DF1[n] for n in nodes])
print(f"DF1 over {len(nodes)} nodes : min {_d.min():.4f} max {_d.max():.4f} | "
      f"distinct {len(set(np.round(_d,6)))}")

In [ ]:
# ----------------------------- Hop rings and groundings -----------------------------
GU = nx.Graph(); GU.add_nodes_from(nodes); GU.add_edges_from(links)
_node_hop = {n: min(nx.shortest_path_length(GU, n, e) for e in DOWN) for n in nodes}

def link_hop(l):
    """0 for the down link itself, else 1 + min endpoint distance to an endpoint of the event."""
    return 0 if key(l) == DOWN else 1 + min(_node_hop[x] for x in l)

HOP   = {l: link_hop(l) for l in links}
RINGS = defaultdict(list)
for l in links:
    RINGS[HOP[l]].append(l)
RINGS = {r: sorted(v) for r, v in sorted(RINGS.items())}

# --- groundings: geography (lon, lat) + demand level (act0), standardised ---
_xy   = np.array([raw_nodes[n] for n in nodes], dtype=np.float64)
_xy   = (_xy - _xy.mean(0)) / _xy.std(0)
_a0   = np.array([act0[n] for n in nodes], dtype=np.float64)
_a0   = _a0 / _a0.max()
NODE_FEAT = {n: np.concatenate([_xy[i], [_a0[i]]]) for i, n in enumerate(nodes)}
NODE_DIM  = 3
LINK_DIM  = 2 * NODE_DIM

print("hop rings from L1:", {r: len(v) for r, v in RINGS.items()})
for r, v in RINGS.items():
    rr = np.array([RHO[l] for l in v])
    print(f"  hop {r}: {len(v):>2} links, rho in [{rr.min():.3f}, {rr.max():.3f}]")
print("\nSpearman(hop, rho) over all links = "
      f"{spearmanr([HOP[l] for l in links], [RHO[l] for l in links]).statistic:.3f}"
      "   <- the topological rule alone")

In [ ]:
# ----------------------------- Reveal policies -----------------------------
# The event link L1 is always revealed: it is the observation that defines the scenario, not an
# inference target (rho = 0 there by construction, so scoring it would inflate every metric).
def reveal_random(frac, seed):
    rng  = np.random.default_rng(seed)
    pool = [l for l in links if l != DOWN]
    k    = max(0, int(round(frac * len(links))) - 1)
    idx  = rng.permutation(len(pool))[:k]
    revealed = [DOWN] + [pool[i] for i in sorted(idx)]
    return revealed, [l for l in links if l not in set(revealed)]

def reveal_hop(frac, seed, min_hold=None):
    """Mode A: neighbourhood-first, but a share of every ring is reserved as held-out.

    Pure nearest-first reveal is degenerate at 50 %: hops 0-2 alone fill the budget, leaving a
    held-out set in which 11 of 13 links have rho >= 0.96, so the rank metrics would be computed
    against a near-constant target. Reserving from each ring keeps the held-out target graded
    while preserving the outward-from-the-fault ordering of what is revealed."""
    if min_hold is None:
        min_hold = CONFIG["min_hold_per_ring"]
    rng      = np.random.default_rng(seed)
    reserved = set()
    for r, ring in RINGS.items():
        if r == 0:
            continue                                     # ring 0 is the event link itself
        k = max(1, int(math.floor(min_hold * len(ring))))
        pick = rng.permutation(len(ring))[:k]
        reserved |= {ring[i] for i in pick}
    pool = [l for l in links if l != DOWN and l not in reserved]
    # nearest ring first; random order within a ring so the boundary is not alphabetical
    order = rng.permutation(len(pool))
    pool  = [pool[i] for i in order]
    pool.sort(key=lambda l: HOP[l])
    k = max(0, int(round(frac * len(links))) - 1)
    if k > len(pool):
        raise ValueError(f"reveal fraction {frac} incompatible with min_hold_per_ring="
                         f"{min_hold}: need {k} links but only {len(pool)} are unreserved. "
                         "Lower min_hold_per_ring or the reveal fraction.")
    revealed = [DOWN] + pool[:k]
    return revealed, [l for l in links if l not in set(revealed)]

REVEAL = {"hop": reveal_hop, "random": reveal_random}

for mode in CONFIG["modes"]:
    for frac in CONFIG["reveal_fracs"]:
        rv, hd = REVEAL[mode](frac, 0)
        rr = np.array([RHO[l] for l in hd])
        rings_hit = sorted({HOP[l] for l in hd})
        print(f"{mode:<7} frac={frac}: revealed {len(rv):>2}, held-out {len(hd):>2}, "
              f"held-out rho in [{rr.min():.3f}, {rr.max():.3f}], rings {rings_hit}")

In [ ]:
# ----------------------------- Metrics (all rank-based) -----------------------------
# The predicted quantity is a fuzzy truth value: a degree, not a calibrated estimate of rho.
# Absolute-error metrics would therefore measure the sigmoid's scale rather than the inference,
# so every metric below depends on the *ordering* of the predictions only. Where a threshold is
# unavoidable it is applied to the ground truth ("what counts as impacted"), which is a semantic
# choice, and never to the model output.
def evaluate(pred, held, cfg):
    """pred: {link: predicted truth value of Link (health, 1 = unaffected)}."""
    y = np.array([RHO[l] for l in held], dtype=float)
    p = np.array([pred[l] for l in held], dtype=float)
    out = {"n_held": len(held)}

    out["spearman"] = spearmanr(p, y).statistic if len(y) > 2 and len(set(y)) > 1 else np.nan

    for tau in cfg["tau_grid"]:
        lab = (y < tau).astype(int)                      # 1 = impacted
        col = f"auroc@{tau}"
        out[col] = roc_auc_score(lab, -p) if 0 < lab.sum() < len(lab) else np.nan

    k = min(cfg["topk"], len(held))
    if k > 0 and len(set(y)) > 1:
        true_worst = set(np.argsort(y, kind="stable")[:k])
        pred_worst = set(np.argsort(p, kind="stable")[:k])
        out[f"top{cfg['topk']}"] = len(true_worst & pred_worst) / k
    else:
        out[f"top{cfg['topk']}"] = np.nan
    return out

In [ ]:
# ----------------------------- Predicate model, axioms, training -----------------------------
class PredMLP(torch.nn.Module):
    def __init__(self, in_dim, hidden):
        super().__init__()
        sizes = (in_dim,) + tuple(hidden) + (1,)
        self.elu, self.sigmoid = torch.nn.ELU(), torch.nn.Sigmoid()
        self.linear_layers = torch.nn.ModuleList(
            [torch.nn.Linear(sizes[i-1], sizes[i]) for i in range(1, len(sizes))])
    def forward(self, *x):
        x = list(x)
        x = x[0] if len(x) == 1 else torch.cat(x, dim=1)
        for layer in self.linear_layers[:-1]:
            x = self.elu(layer(x))
        return self.sigmoid(self.linear_layers[-1](x))

Not     = ltn.Connective(ltn.fuzzy_ops.NotStandard())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall  = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
SatAgg  = ltn.fuzzy_ops.SatAgg()

def _pair_feat(a, b):
    return np.concatenate([NODE_FEAT[a], NODE_FEAT[b]])

def _T(arr):
    return torch.tensor(np.asarray(arr, dtype=np.float32), device=device)

def train_model(revealed, seed, cfg, use_logic=True, use_node=True):
    """Graded supervision + injected axioms.

    Targets are ratios in [0,1] and the predicates output truth values in [0,1], so supervision is
    a direct regression term on the revealed facts; the axioms enter as the usual 1 - SatAgg logic
    loss. Encoding graded labels as weighted assert/negate pairs inside SatAgg would be equivalent
    in spirit but harder to state precisely in the paper.

    use_logic=False, use_node=False gives the feature-only ablation: identical architecture,
    identical features, identical revealed supervision, no axioms and no node knowledge."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    Link_m = PredMLP(LINK_DIM, cfg["hidden"]).to(device)
    Node_m = PredMLP(NODE_DIM, cfg["hidden"]).to(device)
    Link, Node = ltn.Predicate(model=Link_m), ltn.Predicate(model=Node_m)

    # rho is undirected, so both orientations of a revealed link are supervised; this also gives
    # the symmetry axiom consistent evidence rather than leaving it to fight the data.
    X_link = _T([_pair_feat(a, b) for (a, b) in revealed] +
                [_pair_feat(b, a) for (a, b) in revealed])
    y_link = _T([RHO[l] for l in revealed] * 2)
    X_node = _T([NODE_FEAT[n] for n in nodes])
    y_node = _T([DF1[n] for n in nodes])

    nvar = ltn.Variable("n", _T([NODE_FEAT[n] for n in nodes]))
    mvar = ltn.Variable("m", _T([NODE_FEAT[n] for n in nodes]))

    params = list(Link_m.parameters()) + (list(Node_m.parameters()) if use_node else [])
    opt = torch.optim.Adam(params, lr=cfg["lr"])

    for _ in range(cfg["epochs"]):
        opt.zero_grad()
        sup = ((Link_m(X_link).reshape(-1) - y_link) ** 2).mean()
        if use_node:
            sup = sup + ((Node_m(X_node).reshape(-1) - y_node) ** 2).mean()
        loss = cfg["lambda_sup"] * sup
        if use_logic:
            axioms = [
                Forall(nvar, Not(Link(nvar, nvar)), p=5),                          # R1 anti-reflexive
                Forall([nvar, mvar], Implies(Link(nvar, mvar), Link(mvar, nvar)), p=10),  # R2 symmetry
                # R3 propagation: an impacted node makes its incident links impacted. Defeasible --
                # it holds in tendency, not exactly, which is why it is a soft logical constraint
                # and not a hard label.
                Forall([nvar, mvar], Implies(Not(Node(nvar)), Not(Link(nvar, mvar))), p=5),
            ]
            loss = loss + (1.0 - SatAgg(*axioms))
        loss.backward(); opt.step()

    Link_m.eval()
    with torch.no_grad():
        X = _T([_pair_feat(a, b) for (a, b) in links])
        Y = _T([_pair_feat(b, a) for (a, b) in links])
        p = 0.5 * (Link_m(X).reshape(-1) + Link_m(Y).reshape(-1))   # symmetrised readout
        return {l: float(p[i]) for i, l in enumerate(links)}

In [ ]:
# ----------------------------- Baselines -----------------------------
# All predict the same quantity as the model: link "health" in [0,1], higher = less impacted.
_maxhop = max(HOP.values())
_w0u    = {l: w0[l] + w0[l[::-1]] for l in links}
_w0max  = max(_w0u.values())

def base_hop(revealed, seed, cfg):
    """The injected topological rule alone: impact decays with hop distance from the event."""
    return {l: HOP[l] / _maxhop for l in links}

def base_w0(revealed, seed, cfg):
    """Demand alone, no logic: the busiest links are guessed to be the most impacted."""
    return {l: 1.0 - _w0u[l] / _w0max for l in links}

def base_random(revealed, seed, cfg):
    rng = np.random.default_rng(10_000 + seed)
    return {l: float(rng.random()) for l in links}

def model_full(revealed, seed, cfg):
    return train_model(revealed, seed, cfg, use_logic=True,  use_node=True)

def model_feature_only(revealed, seed, cfg):
    return train_model(revealed, seed, cfg, use_logic=False, use_node=False)

PREDICTORS = {
    "LTN (axioms + node)": model_full,
    "feature-only MLP":    model_feature_only,
    "hop distance":        base_hop,
    "load w0":             base_w0,
    "random":              base_random,
}
REFERENCE = "LTN (axioms + node)"

In [ ]:
# ----------------------------- Sweep -----------------------------
rows = []
total = len(CONFIG["modes"]) * len(CONFIG["reveal_fracs"]) * len(CONFIG["seeds"])
done  = 0
for mode in CONFIG["modes"]:
    for frac in CONFIG["reveal_fracs"]:
        for seed in CONFIG["seeds"]:
            revealed, held = REVEAL[mode](frac, seed)

            # --- protocol guards: the held-out targets must never be visible to training ---
            assert not (set(revealed) & set(held)),        "revealed / held-out overlap"
            assert set(revealed) | set(held) == set(links), "split does not cover the link set"
            assert DOWN in revealed and DOWN not in held,   "the event link must be given, not scored"
            assert len(held) >= 3,                          "held-out set too small to rank"

            for mname, fn in PREDICTORS.items():
                pred = fn(revealed, seed, CONFIG)
                rows.append(dict(mode=mode, frac=frac, seed=seed, model=mname,
                                 n_revealed=len(revealed), demand_src=DEMAND_SOURCE,
                                 **evaluate(pred, held, CONFIG)))
            done += 1
            if done % 5 == 0 or done == total:
                print(f"[{done}/{total}] mode={mode} frac={frac} seed={seed}")

results = pd.DataFrame(rows)
print("collected", len(results), "rows")
results.head()

In [ ]:
# ----------------------------- Aggregate -----------------------------
# Index columns with results["..."] rather than attribute access: results.mode would resolve to
# DataFrame.mode (the statistical mode method), silently yielding a scalar comparison and an
# empty selection.
PRIMARY = "spearman"
METRIC_COLS = [PRIMARY] + [f"auroc@{t}" for t in CONFIG["tau_grid"]] + [f"top{CONFIG['topk']}"]

agg = (results.groupby(["mode", "frac", "model"])[METRIC_COLS]
              .agg(["mean", "std"]).round(3))
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print(agg[PRIMARY])

def _pick(model, mode, frac, col):
    m = ((results["model"] == model) & (results["mode"] == mode) & (results["frac"] == frac))
    return results.loc[m].set_index("seed")[col]

def _fmt(model, mode, frac, col):
    s = _pick(model, mode, frac, col).dropna()
    return f"{s.mean():.3f} +/- {s.std():.3f}" if len(s) else "n/a"

def _paired_p(model, mode, frac, col):
    """One-sided Wilcoxon signed-rank, REFERENCE vs `model`, paired by seed on the same split."""
    a = _pick(REFERENCE, mode, frac, col); b = _pick(model, mode, frac, col)
    d = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    d = d[d.a != d.b]
    if len(d) < 5:
        return "n/a"
    return f"{wilcoxon(d.a, d.b, alternative='greater').pvalue:.1e}"

summary = pd.DataFrame([
    dict(Mode=mode, Reveal=f"{int(frac*100)}%", Model=model,
         Spearman=_fmt(model, mode, frac, PRIMARY),
         **{f"AUROC@{t}": _fmt(model, mode, frac, f"auroc@{t}") for t in CONFIG["tau_grid"]},
         **{f"Top{CONFIG['topk']}": _fmt(model, mode, frac, f"top{CONFIG['topk']}")},
         p_vs_ref=("--" if model == REFERENCE else _paired_p(model, mode, frac, PRIMARY)))
    for mode in CONFIG["modes"] for frac in CONFIG["reveal_fracs"] for model in PREDICTORS
])
assert summary["Spearman"].ne("n/a").all(), "empty selection -- check column indexing"
summary

In [ ]:
# ----------------------------- Figure: rank fidelity vs reveal fraction -----------------------------
import matplotlib
import matplotlib.pyplot as plt

STYLE = {
    "LTN (axioms + node)": dict(color="tab:blue",   marker="o", lw=2.0, zorder=5),
    "feature-only MLP":    dict(color="tab:orange", marker="s", lw=1.6, ls="--"),
    "hop distance":        dict(color="tab:green",  marker="^", lw=1.6, ls="-."),
    "load w0":             dict(color="tab:red",    marker="v", lw=1.2, ls=":"),
    "random":              dict(color="grey",       marker="x", lw=1.2, ls=":"),
}
TITLE = {"hop": "Mode A -- neighbourhood-first (ring-stratified)",
         "random": "Mode B -- random reveal"}

fig, axes = plt.subplots(1, len(CONFIG["modes"]), figsize=(6.2*len(CONFIG["modes"]), 4.4),
                         sharey=True, squeeze=False)
for ax, mode in zip(axes[0], CONFIG["modes"]):
    for mname in PREDICTORS:
        mu = [_pick(mname, mode, f, PRIMARY).mean() for f in CONFIG["reveal_fracs"]]
        sd = [_pick(mname, mode, f, PRIMARY).std()  for f in CONFIG["reveal_fracs"]]
        x  = [int(f*100) for f in CONFIG["reveal_fracs"]]
        ax.errorbar(x, mu, yerr=sd, capsize=3, label=mname, **STYLE[mname])
    ax.axhline(0.0, color="black", lw=0.8, alpha=0.5)
    ax.set_xlabel("share of link facts revealed (%)")
    ax.set_title(TITLE.get(mode, mode), fontsize=10)
    ax.set_xticks([int(f*100) for f in CONFIG["reveal_fracs"]])
    ax.grid(alpha=0.3)
axes[0][0].set_ylabel(r"Spearman $\rho$ vs held-out $\rho(l)$")
axes[0][-1].legend(fontsize=8, loc="lower right", framealpha=0.9)
fig.suptitle(f"Held-out link inference after Link_Down(Berlin, Hamburg), "
             f"{len(CONFIG['seeds'])} seeds (mean $\\pm$ std)", fontsize=11)
fig.tight_layout()
os.makedirs("img", exist_ok=True)
fig.savefig(os.path.join("img", "case_study_kt1_spearman.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# ----------------------------- Save -----------------------------
results.to_csv("case_study_kt1_raw.csv", index=False)
summary.to_csv("case_study_kt1_summary.csv", index=False)

# Ground-truth table for the paper: per-link targets with hop ring, sorted by impact.
gt_table = pd.DataFrame([
    dict(link=link_name[l], u=l[0], v=l[1], hop=HOP[l], rho=round(RHO[l], 4),
         w0=round(_w0u[l], 2))
    for l in sorted(links, key=lambda x: RHO[x])
])
gt_table.to_csv("case_study_kt1_groundtruth.csv", index=False)
print("saved case_study_kt1_{raw,summary,groundtruth}.csv and img/case_study_kt1_spearman.pdf")
gt_table.head(10)

## Notes for the write-up

* **Scope.** One event (L1, Berlin-Hamburg), matching Fig. 13. Variability is over seeds and over
  the revealed subset, not over scenarios. This is deliberate: the claim concerns what the model
  infers from partial information about *this* configuration, and a 26-scenario sweep would recast
  a case study as a benchmark, inviting comparisons against methods the paper does not propose.

* **Why rank metrics.** The predicted quantity is a fuzzy truth value -- a degree of membership,
  not a calibrated estimate of $\rho$. Absolute error would therefore measure the sigmoid's scale
  rather than the quality of the inference. Spearman is the primary metric; the AUROC columns apply
  a threshold to the *ground truth* ("what counts as impacted"), which is a semantic choice, never
  to the model output. Top-$k$ is included because it is the form an operator would act on.

* **Why accuracy is absent.** A binary affected/unaffected label over 26 links leaves roughly two
  positives per held-out set, so a constant "nothing is affected" predictor scores about 0.8. Any
  accuracy figure would report the class prior. The graded targets remove the problem at its
  source rather than correcting for it afterwards.

* **The baselines carry the argument.** Hop distance from L1 -- the injected topological rule with
  no learning at all -- already reaches Spearman $\approx 0.81$ against $\rho$ on the full link set.
  Any claim for the model must be stated relative to that, and to the feature-only MLP, which sees
  the same features and the same revealed supervision with the axioms and `Node` removed. The
  interesting comparison is therefore not model-vs-random but **model-vs-rule** and
  **model-vs-same-model-without-logic**, reported with a seed-paired Wilcoxon test.

* **What the model can add.** Hop distance orders the rings but is constant within a ring, whereas
  $\rho$ is not: at $\beta = 0.5$ the hop-3 ring spans $[0.582, 1.000]$, with L25
  (Nuernberg-Stuttgart) far below its ring-mates because of where traffic reroutes. Within-ring
  ordering is exactly what topological reasoning alone cannot supply and what demand knowledge can.
  If the model's advantage over hop distance is not significant, that should be reported plainly;
  the fallback claim -- that the model reproduces the rule while additionally yielding graded
  per-link degrees from partial observations -- is still a legitimate case-study result.

* **Non-circularity, stated precisely.** Held-out $\rho$ values never enter the loss, and the guards
  in the sweep assert it. Two honest qualifications: KT$_1$ is a *synthetic* ground truth built from
  real traffic, in the same spirit as P1/P2, so the claim is recovery of a known construction from
  partial observation rather than prediction of a measured outcome; and KT$_1$'s tier-2 set is
  defined via the 1-hop rule, so $\rho$ is not fully independent of the injected axioms. What is
  independent is the *magnitude* of each link's ratio, which comes from the measured demand matrix
  and is what the rank metrics score.

* **$\beta$ sensitivity.** $\beta \in \{0.3, 0.5, 0.7\}$ leaves the structure of $\rho$ unchanged
  (23/26 links below 1, 21 distinct values, largest tie block 4) and only rescales DF$_1$, so the
  conclusions do not hinge on $\beta = 0.5$. Re-running with `CONFIG["beta"]` altered confirms this
  cheaply.

* **Data.** Real SNDlib matrix `demandMatrix-germany17-DFN-5min-20050215-0000.txt` (251 directed
  demands, total volume 3512.47 -- identical to `demand_factor.ipynb`), routed on the same $w_g$
  metric so both notebooks agree on every path. Nine consecutive 5-minute matrices are available in
  the archive; repeating the study across them would give a traffic-variability check if a reviewer
  questions the single snapshot.

* **Cost.** `len(modes) x len(reveal_fracs) x len(seeds)` splits, two trainings each (full model and
  feature-only ablation); the other three baselines are closed-form. Set `quick_test=True` first.